# Exercício: Classificação Binária
## Objetivos: Exercitar os conceitos referente à classificação binária.

In [1]:
import duckdb
import unicodedata
import re
from datasketch import MinHash, MinHashLSH
from tqdm import tqdm
from joblib import Parallel, delayed
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score, make_scorer
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from gensim.models import Word2Vec
from scipy.sparse import hstack
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.compose import ColumnTransformer
from imblearn.pipeline import Pipeline as ImbPipeline
from tqdm import tqdm

### 1. Crie um arquivo Jupyter Notebook e realize as seguintes operações:

a. Ler o dataset fakeTelegram.BR_2022.csv

b. Remova os trava-zaps, as linhas repetidas (duplicadas) e textos com menos de 5 palavras.

c. Agrupe as linhas com postagens iguais ou extremamente semelhantes. Aqui você pode utilizar uma métrica de semelhança de textos. Crie uma variável para representar a quantidade de vezes que a mensagem foi compartilhada. Observe que ao agrupar linhas que possuem a “mesma” postagem (texto), você deve escolher como valor para as variáveis data e hora da postagem, os valores da cópia mais antiga.

d. Você pode criar novos atributos numéricos, tais como: quantidade de palavras, quantidade de caracteres etc.

In [2]:
conn = duckdb.connect()

telegram_data = conn.read_csv("../data/fakeTelegram.BR_2022.csv")

query = """
    SELECT * FROM telegram_data
"""

df = conn.execute(query).fetchdf()

In [3]:
df.head()

,date_message,id_member_anonymous,id_group_anonymous,media,media_type,media_url,has_media,has_media_url,trava_zap,text_content_anonymous,dataset_info_id,date_system,score_sentiment,score_misinformation,id_message,message_type,messenger,media_name,media_md5
0,2022-10-05 06:25:04,1078cc958f0febe28f4d03207660715f,12283e08a2eb5789201e105b34489ee7,None,None,None,False,False,False,Então é Fato Renato o áudio que eu ouvi no wha...,5,2022-10-05 06:25:28.863641,0.0000,NaN,16385,Texto,telegram,None,None
1,2022-10-05 06:25:08,None,12283e08a2eb5789201e105b34489ee7,None,None,None,False,False,False,"Saiu no YouTube do presidente a 8 horas atrás,...",5,2022-10-05 06:25:28.926311,0.0644,NaN,16386,Texto,telegram,None,None
2,2022-10-05 06:26:28,92a2d8fd7144074f659d1d29dc3751da,9f2d7394334eb224c061c9740b5748fc,None,None,None,False,False,False,"É isso, nossa parte já foi quase toda feita. N...",5,2022-10-05 06:26:29.361949,-0.3551,0.157242,16366,Texto,telegram,None,None
3,2022-10-05 06:27:28,d60aa38f62b4977426b70944af4aff72,c8f2de56550ed0bf85249608b7ead93d,94dca4cda503100ebfda7ce2bcc060eb.jpg,image/jpg,None,True,False,False,GENTE ACHEI ELES EM UMA SEITA MAÇONÁRICA,5,2022-10-05 06:27:29.935624,0.0000,NaN,19281,Imagem,telegram,None,94dca4cda503100ebfda7ce2bcc060eb
4,2022-10-05 06:27:44,cd6979b0b5265f08468fa1689b6300ce,e56ec342fc599ebb4ed89655eb6f03aa,5ad5c8bbe9da93a37fecf3e5aa5b0637.jpg,image/jpg,None,True,False,False,None,5,2022-10-05 06:28:29.316325,NaN,NaN,507185,Imagem,telegram,None,5ad5c8bbe9da93a37fecf3e5aa5b0637


In [4]:
df.shape

(557586, 19)

In [5]:
df = conn.execute(f"""
    SELECT * 
    FROM telegram_data
    WHERE trava_zap IS NOT TRUE
""").fetch_df()

In [6]:
df.shape

(557570, 19)

In [7]:
df_ = conn.execute("SELECT DISTINCT * FROM df").fetch_df()

In [8]:
query = """
SELECT *
FROM df_
WHERE array_length(string_split(text_content_anonymous, ' ')) >= 5
"""

df = conn.execute(query).fetch_df()

df.shape

(336944, 19)

In [9]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    
    # 1. Converte para minúsculas
    text = text.lower()
    
    # 2. Remover URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    
    # 3. Remover caracteres árabes
    text = re.sub(r'[\u0600-\u06FF]+', '', text)
    
    # 4. Remover acentos (Normalização)
    text = unicodedata.normalize('NFKD', text)\
           .encode('ascii', 'ignore')\
           .decode('utf-8')
    
    # 5. Remover pontuação e caracteres especiais
    text = re.sub(r'[^\w\s]', '', text)
    
    # 6. Remover emojis 
    emoji_pattern = re.compile("["
        u"\U0001F600-\U0001F64F"  # emoticons
        u"\U0001F300-\U0001F5FF"  # símbolos & pictogramas
        u"\U0001F680-\U0001F6FF"  # transporte & símbolos
        u"\U0001F1E0-\U0001F1FF"  # bandeiras (iOS)
                           "]+", flags=re.UNICODE)
    text = emoji_pattern.sub(r'', text)
    
    # 7. Remover espaços extras (no início, fim e múltiplos espaços no meio)
    text = ' '.join(text.split())
    
    return text

df['cleaned_text'] = df['text_content_anonymous'].apply(clean_text)

In [10]:
def get_shingles(text, k=5):
    """Cria um conjunto de shingles de tamanho k a partir de um texto."""
    if not isinstance(text, str) or len(text) < k:
        return set()
    return set(text[i:i+k] for i in range(len(text) - k + 1))

def create_minhash(text, num_perm, shingle_size):
    mh = MinHash(num_perm=num_perm)
    shingles = get_shingles(text, k=shingle_size)
    if not shingles:
        return None
    for shingle in shingles:
        mh.update(shingle.encode('utf-8'))
    return mh

In [11]:
print("Iniciando agrupamento de textos semelhantes...")

JACCARD_THRESHOLD = 0.7
NUM_PERMUTATIONS = 128
SHINGLE_SIZE = 5

texts = df['cleaned_text'].tolist()

print("Etapa 1 de 4: Criando MinHashes em paralelo...")
minhash_list = Parallel(n_jobs=-1)(
    delayed(create_minhash)(text, NUM_PERMUTATIONS, SHINGLE_SIZE) 
    for text in tqdm(texts, desc="Criando assinaturas")
)

# Remapeia a lista para o dicionário, o que é rápido
minhashes = {idx: mh for idx, mh in enumerate(minhash_list) if mh is not None}

print("\nEtapa 2 de 4: Indexando MinHashes no LSH...")
lsh = MinHashLSH(threshold=JACCARD_THRESHOLD, num_perm=NUM_PERMUTATIONS)
for idx, mh in tqdm(minhashes.items(), desc="Indexando no LSH"):
    lsh.insert(idx, mh)

print("\nEtapa 3 de 4: Consultando LSH e agrupando...")
groups = []
used = set()
for idx in tqdm(minhashes, desc="Consultando e agrupando"):
    if idx not in used:
        similar_indices = lsh.query(minhashes[idx])
        groups.append(similar_indices)
        used.update(similar_indices)

print(f"\nAgrupamento concluído. Foram encontrados {len(groups)} grupos de mensagens.")


print("\nEtapa 4 de 4: Consolidando resultados...")
grouped_data = []
for group in tqdm(groups, desc="Consolidando grupos"):
    group_df = df.iloc[list(group)]
    oldest = group_df.loc[group_df['date_message'].idxmin()]
    count = len(group_df)
    new_row = oldest.to_dict()
    new_row['shared_count'] = count
    grouped_data.append(new_row)

result_df = pd.DataFrame(grouped_data)
result_df = result_df.sort_values(by='shared_count', ascending=False)

print("\nProcesso finalizado com sucesso!")

Iniciando agrupamento de textos semelhantes...
Etapa 1 de 4: Criando MinHashes em paralelo...


Criando assinaturas: 100%|██████████| 336944/336944 [02:29<00:00, 2253.54it/s]



Etapa 2 de 4: Indexando MinHashes no LSH...


Indexando no LSH: 100%|██████████| 336652/336652 [00:13<00:00, 25221.76it/s]



Etapa 3 de 4: Consultando LSH e agrupando...


Consultando e agrupando: 100%|██████████| 336652/336652 [00:04<00:00, 75839.43it/s]



Agrupamento concluído. Foram encontrados 183779 grupos de mensagens.

Etapa 4 de 4: Consolidando resultados...


Consolidando grupos: 100%|██████████| 183779/183779 [01:03<00:00, 2887.16it/s]



Processo finalizado com sucesso!


In [12]:
def extract_features(text):
    if not isinstance(text, str):
        return {
            'word_count': 0,
            'char_count': 0,
            'unique_word_ratio': 0,
            'avg_word_length': 0
        }
    
    words = text.split()
    word_count = len(words)
    char_count = len(text)
    
    if word_count > 0:
        unique_words = set(words)
        unique_word_ratio = len(unique_words) / word_count
        avg_word_length = sum(len(word) for word in words) / word_count
    else:
        unique_word_ratio = 0
        avg_word_length = 0
    
    return {
        'word_count': word_count,
        'char_count': char_count,
        'unique_word_ratio': unique_word_ratio,
        'avg_word_length': avg_word_length
    }

features = result_df['cleaned_text'].apply(lambda x: pd.Series(extract_features(x)))

result_df = pd.concat([result_df, features], axis=1)

print("Características numéricas adicionadas com sucesso ao result_df.")
print(result_df[['cleaned_text', 'shared_count', 'word_count', 'char_count', 'unique_word_ratio', 'avg_word_length']].head())

Características numéricas adicionadas com sucesso ao result_df.
                                            cleaned_text  shared_count  \
13237  this community was blocked in brazil following...         17422   
2002   welcome abrarov djahongir user professional to...          4937   
2218   welcome helen user professional tool for manag...          3028   
2052   welcome helen user professional tool for manag...          2751   
2049   welcome helen user professional tool for manag...          2275   

       word_count  char_count  unique_word_ratio  avg_word_length  
13237        15.0        93.0                1.0         5.266667  
2002         10.0        77.0                1.0         6.800000  
2218          9.0        65.0                1.0         6.333333  
2052          9.0        65.0                1.0         6.333333  
2049          9.0        65.0                1.0         6.333333  


Utilizando os dados referente a postagens no Telegram, crie um modelo preditivo (classificador binário) para classificar uma mensagem em duas classes possíveis: “viral” (classe positiva) ou “não viral” (classe negativa). Para rotular as mensagens únicas (agrupadas) nas classes, “viral” e “não viral”, utilize a seguinte estratégia: Calcule um limiar (threshold). Por exemplo, mediana do número de compartilhamentos mais dois desvios padrões. As mensagens com quantidade de compartilhamentos maiores ou iguais ao limiar definido devem ser rotuladas como “virais”. As demais mensagens devem ser rotuladas como “não virais”.

A avaliação experimental deverá considerar:

e. O algoritmo de classificação: regressão logística, árvore de decisão e uma estratégia baseada em “ensemble”;

f. Regularização: Com regularização (Ridge, Lasso ou ElasticNet) e sem regularização;

g. Normalização dos dados: sem normalização, Z-Score, Min-Max (OPCIONAL); 

h. Pré-processamento de dados: sem pré-processamento e com pré-processamento;

i. Embedding: BOW, TF-IDF, Word2Vec;

j. N-Gramas: unigramas, bigramas, trigramas;

k. Treinamento, Validação e Teste: Outer K-Fold Cross-Validation;

In [13]:
median_shares = result_df['shared_count'].median()
std_dev_shares = result_df['shared_count'].std()
viral_threshold = median_shares + (2 * std_dev_shares)

print(f"Análise para definir 'viral':")
print(f"  - Mediana de compartilhamentos: {median_shares:.2f}")
print(f"  - Desvio Padrão dos compartilhamentos: {std_dev_shares:.2f}")
print(f"  - Limiar de Viralização (Mediana + 2*DP): {viral_threshold:.2f}")
print("-" * 30)

result_df['is_viral'] = np.where(result_df['shared_count'] >= viral_threshold, 1, 0)

class_distribution = result_df['is_viral'].value_counts(normalize=True) * 100

print("Distribuição das Classes:")
print(f"  - Não Viral (0): {class_distribution.get(0, 0):.2f}%")
print(f"  - Viral (1):     {class_distribution.get(1, 0):.2f}%")


print("\nExemplo de dados rotulados:")
print(result_df[['shared_count', 'is_viral']].sort_values(by='shared_count', ascending=False).head())

Análise para definir 'viral':
  - Mediana de compartilhamentos: 1.00
  - Desvio Padrão dos compartilhamentos: 47.03
  - Limiar de Viralização (Mediana + 2*DP): 95.06
------------------------------
Distribuição das Classes:
  - Não Viral (0): 99.90%
  - Viral (1):     0.10%

Exemplo de dados rotulados:
       shared_count  is_viral
13237         17422         1
2002           4937         1
2218           3028         1
2052           2751         1
2049           2275         1


In [14]:
!pip install imbalanced-learn

# e. O algoritmo de classificação: regressão logística, árvore de decisão e uma estratégia baseada em “ensemble”;
# f. Regularização: Com regularização (Ridge, Lasso ou ElasticNet) e sem regularização;

In [15]:
features = [
    'word_count', 
    'char_count', 
    'avg_word_length',
    'unique_word_ratio',
]
X = result_df[features]
y = result_df['is_viral']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

print(f"Distribuição original no treino: {y_train.value_counts().to_dict()}")

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print(f"Distribuição após SMOTE no treino: {y_train_resampled.value_counts().to_dict()}")

models = {
    "Reg. Log. - Sem Regularização": LogisticRegression(penalty=None, solver='saga', random_state=42, max_iter=1000),
    "Reg. Log. - L2 (Ridge)": LogisticRegression(penalty='l2', C=1.0, solver='saga', random_state=42, max_iter=1000),
    "Reg. Log. - L1 (Lasso)": LogisticRegression(penalty='l1', C=1.0, solver='saga', random_state=42, max_iter=1000),
    "Árvore de Decisão": DecisionTreeClassifier(random_state=42),
    "Random Forest (Ensemble)": RandomForestClassifier(random_state=42)
}

for name, model in models.items():
    print(f"\n--- Treinando e Avaliando: {name} ---")
    
    model.fit(X_train_resampled, y_train_resampled)
    
    y_pred = model.predict(X_test)
    
    print(classification_report(y_test, y_pred, target_names=["Não Viral (0)", "Viral (1)"]))

    if 'Reg. Log.' in name:
        print("   Coeficientes (importância das features):")
        coefs = pd.Series(model.coef_[0], index=features)
        print(coefs.sort_values(ascending=False))

Distribuição original no treino: {0: 137691, 1: 143}
Distribuição após SMOTE no treino: {0: 137691, 1: 137691}

--- Treinando e Avaliando: Reg. Log. - Sem Regularização ---


c:\Users\Pedro\miniforge3\envs\cd\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


               precision    recall  f1-score   support

Não Viral (0)       1.00      0.61      0.76     45897
    Viral (1)       0.00      0.56      0.00        48

     accuracy                           0.61     45945
    macro avg       0.50      0.59      0.38     45945
 weighted avg       1.00      0.61      0.76     45945

   Coeficientes (importância das features):
avg_word_length      0.220012
char_count           0.009654
word_count          -0.054717
unique_word_ratio   -0.463061
dtype: float64

--- Treinando e Avaliando: Reg. Log. - L2 (Ridge) ---


c:\Users\Pedro\miniforge3\envs\cd\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


               precision    recall  f1-score   support

Não Viral (0)       1.00      0.61      0.76     45897
    Viral (1)       0.00      0.56      0.00        48

     accuracy                           0.61     45945
    macro avg       0.50      0.59      0.38     45945
 weighted avg       1.00      0.61      0.76     45945

   Coeficientes (importância das features):
avg_word_length      0.220006
char_count           0.009654
word_count          -0.054717
unique_word_ratio   -0.463033
dtype: float64

--- Treinando e Avaliando: Reg. Log. - L1 (Lasso) ---


c:\Users\Pedro\miniforge3\envs\cd\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


               precision    recall  f1-score   support

Não Viral (0)       1.00      0.61      0.76     45897
    Viral (1)       0.00      0.56      0.00        48

     accuracy                           0.61     45945
    macro avg       0.50      0.59      0.38     45945
 weighted avg       1.00      0.61      0.76     45945

   Coeficientes (importância das features):
avg_word_length      0.219987
char_count           0.009655
word_count          -0.054718
unique_word_ratio   -0.462940
dtype: float64

--- Treinando e Avaliando: Árvore de Decisão ---
               precision    recall  f1-score   support

Não Viral (0)       1.00      0.99      0.99     45897
    Viral (1)       0.03      0.38      0.05        48

     accuracy                           0.99     45945
    macro avg       0.51      0.68      0.52     45945
 weighted avg       1.00      0.99      0.99     45945


--- Treinando e Avaliando: Random Forest (Ensemble) ---
               precision    recall  f1-score   s

# item g é opcional, pulei nessa lista
# h. Pré-processamento de dados: sem pré-processamento e com pré-processamento;

In [16]:
print("="*20)
print("CENÁRIO 1: SEM PRÉ-PROCESSAMENTO NUMÉRICO")
print("="*20)

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

models_sem_preproc = {
    "Regressão Logística": LogisticRegression(random_state=42, max_iter=1000),
    "Árvore de Decisão": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42)
}

for name, model in models_sem_preproc.items():
    print(f"\n--- Avaliando: {name} ---")
    model.fit(X_train_smote, y_train_smote)
    y_pred = model.predict(X_test)
    print(classification_report(y_test, y_pred, target_names=["Não Viral", "Viral"]))


print("\n" + "="*20)
print("CENÁRIO 2: COM PRÉ-PROCESSAMENTO NUMÉRICO")
print("="*20)

def remove_outliers_iqr(X, y):
    X_clean = X.copy()
    y_clean = y.copy()
    for col in X_clean.columns:
        Q1 = X_clean[col].quantile(0.3)
        Q3 = X_clean[col].quantile(0.7)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        mask = (X_clean[col] >= lower_bound) & (X_clean[col] <= upper_bound)
        X_clean = X_clean.loc[mask]
        y_clean = y_clean.loc[mask]
    return X_clean, y_clean

X_train_no_outliers, y_train_no_outliers = remove_outliers_iqr(X_train, y_train)
print(f"Removendo outliers do treino: {len(X_train)} -> {len(X_train_no_outliers)} amostras")

X_train_proc_smote, y_train_proc_smote = smote.fit_resample(X_train_no_outliers, y_train_no_outliers)

pipelines_com_preproc = {
    "Regressão Logística": Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', LogisticRegression(random_state=42, max_iter=1000))
    ]),
    "Árvore de Decisão": Pipeline([ 
        ('classifier', DecisionTreeClassifier(random_state=42))
    ]),
    "Random Forest": Pipeline([
        ('classifier', RandomForestClassifier(random_state=42))
    ])
}

for name, pipeline in pipelines_com_preproc.items():
    print(f"\n--- Avaliando: {name} ---")
    pipeline.fit(X_train_proc_smote, y_train_proc_smote)
    y_pred = pipeline.predict(X_test)
    print(classification_report(y_test, y_pred, target_names=["Não Viral", "Viral"]))


CENÁRIO 1: SEM PRÉ-PROCESSAMENTO NUMÉRICO

--- Avaliando: Regressão Logística ---
              precision    recall  f1-score   support

   Não Viral       1.00      0.74      0.85     45897
       Viral       0.00      0.52      0.00        48

    accuracy                           0.74     45945
   macro avg       0.50      0.63      0.43     45945
weighted avg       1.00      0.74      0.85     45945


--- Avaliando: Árvore de Decisão ---
              precision    recall  f1-score   support

   Não Viral       1.00      0.99      0.99     45897
       Viral       0.03      0.38      0.05        48

    accuracy                           0.99     45945
   macro avg       0.51      0.68      0.52     45945
weighted avg       1.00      0.99      0.99     45945


--- Avaliando: Random Forest ---
              precision    recall  f1-score   support

   Não Viral       1.00      0.99      0.99     45897
       Viral       0.03      0.35      0.05        48

    accuracy                

# i. Embedding: BOW, TF-IDF, Word2Vec;

In [18]:
features_df = result_df['cleaned_text'].apply(lambda x: pd.Series(extract_features(x)))
result_df = pd.concat([result_df, features_df], axis=1)


text_feature = 'cleaned_text'
numeric_features = ['word_count', 'char_count', 'unique_word_ratio', 'avg_word_length']

X = result_df[numeric_features + [text_feature]]
y = result_df['is_viral']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

models_to_run = {
    "Regressão Logística": LogisticRegression(penalty='l2', C=1.0, solver='saga', random_state=42, max_iter=2000),
    "Random Forest": RandomForestClassifier(random_state=42, n_jobs=-1)
}

binary_label_names = ["Não Viral", "Viral"]

print("\n" + "="*60)
print("INICIANDO CENÁRIO 1: TF-IDF (CLASSIFICAÇÃO BINÁRIA)")
print("="*60)

tfidf_vectorizer = TfidfVectorizer(max_features=5000)
X_train_text_tfidf = tfidf_vectorizer.fit_transform(X_train[text_feature])
X_test_text_tfidf = tfidf_vectorizer.transform(X_test[text_feature])

scaler_tfidf = StandardScaler()
X_train_num_scaled_tfidf = scaler_tfidf.fit_transform(X_train[numeric_features])
X_test_num_scaled_tfidf = scaler_tfidf.transform(X_test[numeric_features])

X_train_final_tfidf = hstack([X_train_num_scaled_tfidf, X_train_text_tfidf])
X_test_final_tfidf = hstack([X_test_num_scaled_tfidf, X_test_text_tfidf])

print("Aplicando SMOTE para TF-IDF...")
smote_tfidf = SMOTE(random_state=42)
X_train_resampled_tfidf, y_train_resampled_tfidf = smote_tfidf.fit_resample(X_train_final_tfidf, y_train)
print("SMOTE concluído.")

for name, model in tqdm(models_to_run.items(), desc="Treinando Modelos com TF-IDF"):
    model.fit(X_train_resampled_tfidf, y_train_resampled_tfidf)
    y_pred = model.predict(X_test_final_tfidf)
    print(f"\n--- Resultado para: {name} com TF-IDF ---")
    print(classification_report(y_test, y_pred, target_names=binary_label_names, zero_division=0))


print("\n" + "="*60)
print("INICIANDO CENÁRIO 2: BAG-OF-WORDS (BOW) (CLASSIFICAÇÃO BINÁRIA)")
print("="*60)

bow_vectorizer = CountVectorizer(max_features=5000)
X_train_text_bow = bow_vectorizer.fit_transform(X_train[text_feature])
X_test_text_bow = bow_vectorizer.transform(X_test[text_feature])

scaler_bow = StandardScaler()
X_train_num_scaled_bow = scaler_bow.fit_transform(X_train[numeric_features])
X_test_num_scaled_bow = scaler_bow.transform(X_test[numeric_features])

X_train_final_bow = hstack([X_train_num_scaled_bow, X_train_text_bow])
X_test_final_bow = hstack([X_test_num_scaled_bow, X_test_text_bow])

print("Aplicando SMOTE para BOW...")
smote_bow = SMOTE(random_state=42)
X_train_resampled_bow, y_train_resampled_bow = smote_bow.fit_resample(X_train_final_bow, y_train)
print("SMOTE concluído.")

for name, model in tqdm(models_to_run.items(), desc="Treinando Modelos com BOW"):
    model.fit(X_train_resampled_bow, y_train_resampled_bow)
    y_pred = model.predict(X_test_final_bow)
    print(f"\n--- Resultado para: {name} com BOW ---")
    print(classification_report(y_test, y_pred, target_names=binary_label_names, zero_division=0))


print("\n" + "="*60)
print("INICIANDO CENÁRIO 3: WORD2VEC (CLASSIFICAÇÃO BINÁRIA)")
print("="*60)

tokenized_text_train = [text.split() for text in X_train[text_feature]]
vector_size = 100
print("Treinando modelo Word2Vec...")
w2v_model = Word2Vec(sentences=tokenized_text_train, vector_size=vector_size, window=5, min_count=2, workers=4)
print("Modelo Word2Vec treinado.")

def document_vector(doc, model, num_features):
    doc_vector = np.zeros((num_features,), dtype="float32")
    num_words = 0
    words = doc.split()
    for word in words:
        if word in model.wv:
            num_words += 1
            doc_vector = np.add(doc_vector, model.wv[word])
    if num_words > 0:
        doc_vector = np.divide(doc_vector, num_words)
    return doc_vector

X_train_text_w2v = np.array([document_vector(doc, w2v_model, vector_size) for doc in tqdm(X_train[text_feature], desc="Criando vetores de treino (W2V)")])
X_test_text_w2v = np.array([document_vector(doc, w2v_model, vector_size) for doc in tqdm(X_test[text_feature], desc="Criando vetores de teste (W2V)")])

scaler_w2v = StandardScaler()
X_train_num_scaled_w2v = scaler_w2v.fit_transform(X_train[numeric_features])
X_test_num_scaled_w2v = scaler_w2v.transform(X_test[numeric_features])

X_train_final_w2v = np.hstack([X_train_num_scaled_w2v, X_train_text_w2v])
X_test_final_w2v = np.hstack([X_test_num_scaled_w2v, X_test_text_w2v])

print("Aplicando SMOTE para Word2Vec...")
smote_w2v = SMOTE(random_state=42)
X_train_resampled_w2v, y_train_resampled_w2v = smote_w2v.fit_resample(X_train_final_w2v, y_train)
print("SMOTE concluído.")

for name, model in tqdm(models_to_run.items(), desc="Treinando Modelos com Word2Vec"):
    model.fit(X_train_resampled_w2v, y_train_resampled_w2v)
    y_pred = model.predict(X_test_final_w2v)
    print(f"\n--- Resultado para: {name} com Word2Vec ---")
    print(classification_report(y_test, y_pred, target_names=binary_label_names, zero_division=0))


INICIANDO CENÁRIO 1: TF-IDF (CLASSIFICAÇÃO BINÁRIA)
Aplicando SMOTE para TF-IDF...
SMOTE concluído.


Treinando Modelos com TF-IDF:  50%|█████     | 1/2 [05:54<05:54, 354.77s/it]


--- Resultado para: Regressão Logística com TF-IDF ---
              precision    recall  f1-score   support

   Não Viral       1.00      0.99      0.99     45897
       Viral       0.06      0.69      0.12        48

    accuracy                           0.99     45945
   macro avg       0.53      0.84      0.56     45945
weighted avg       1.00      0.99      0.99     45945



Treinando Modelos com TF-IDF: 100%|██████████| 2/2 [06:18<00:00, 189.17s/it]


--- Resultado para: Random Forest com TF-IDF ---
              precision    recall  f1-score   support

   Não Viral       1.00      1.00      1.00     45897
       Viral       0.25      0.31      0.28        48

    accuracy                           1.00     45945
   macro avg       0.62      0.66      0.64     45945
weighted avg       1.00      1.00      1.00     45945


INICIANDO CENÁRIO 2: BAG-OF-WORDS (BOW) (CLASSIFICAÇÃO BINÁRIA)


...

Aplicando SMOTE para Word2Vec...
SMOTE concluído.


Treinando Modelos com Word2Vec:  50%|█████     | 1/2 [00:56<00:56, 56.16s/it]


--- Resultado para: Regressão Logística com Word2Vec ---
              precision    recall  f1-score   support

   Não Viral       1.00      0.92      0.96     45897
       Viral       0.01      0.81      0.02        48

    accuracy                           0.92     45945
   macro avg       0.51      0.87      0.49     45945
weighted avg       1.00      0.92      0.96     45945



Treinando Modelos com Word2Vec: 100%|██████████| 2/2 [02:29<00:00, 74.82s/it]


--- Resultado para: Random Forest com Word2Vec ---
              precision    recall  f1-score   support

   Não Viral       1.00      1.00      1.00     45897
       Viral       0.26      0.23      0.24        48

    accuracy                           1.00     45945
   macro avg       0.63      0.61      0.62     45945
weighted avg       1.00      1.00      1.00     45945



# j. N-Gramas: unigramas, bigramas, trigramas;

In [19]:
X = result_df[numeric_features + [text_feature]] 
y = result_df['is_viral']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

ngram_configs = {
    "Apenas Unigramas": (1, 1),
    "Unigramas e Bigramas": (1, 2),
    "Unigramas, Bigramas e Trigramas": (1, 3)
}

for name, ngram_range in ngram_configs.items():
    print("="*40)
    print(f"TESTANDO CONFIGURAÇÃO: {name}")
    print("="*40)
    
    print(f"Criando features TF-IDF com ngram_range={ngram_range}...")
    tfidf_vectorizer = TfidfVectorizer(
        ngram_range=ngram_range, 
        max_features=5000
    )
    X_train_text = tfidf_vectorizer.fit_transform(X_train[text_feature])
    X_test_text = tfidf_vectorizer.transform(X_test[text_feature])

    print("Combinando features numéricas e de texto...")
    scaler = StandardScaler()
    X_train_num_scaled = scaler.fit_transform(X_train[numeric_features])
    X_test_num_scaled = scaler.transform(X_test[numeric_features])
    X_train_final = hstack([X_train_num_scaled, X_train_text])
    X_test_final = hstack([X_test_num_scaled, X_test_text])

    print("Rebalanceando o conjunto de treino com SMOTE...")
    smote = SMOTE(random_state=42)
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train_final, y_train)

    print("Treinando o modelo Random Forest...")
    model = RandomForestClassifier(random_state=42)
    model.fit(X_train_resampled, y_train_resampled)
    
    print(f"\n--- Resultado para: {name} ---")
    y_pred = model.predict(X_test_final)
    print(classification_report(y_test, y_pred, target_names=["Não Viral", "Viral"]))

TESTANDO CONFIGURAÇÃO: Apenas Unigramas
Criando features TF-IDF com ngram_range=(1, 1)...
Combinando features numéricas e de texto...
Rebalanceando o conjunto de treino com SMOTE...
Treinando o modelo Random Forest...

--- Resultado para: Apenas Unigramas ---
              precision    recall  f1-score   support

   Não Viral       1.00      1.00      1.00     45897
       Viral       0.25      0.31      0.28        48

    accuracy                           1.00     45945
   macro avg       0.62      0.66      0.64     45945
weighted avg       1.00      1.00      1.00     45945

TESTANDO CONFIGURAÇÃO: Unigramas e Bigramas
Criando features TF-IDF com ngram_range=(1, 2)...
Combinando features numéricas e de texto...
Rebalanceando o conjunto de treino com SMOTE...
Treinando o modelo Random Forest...

--- Resultado para: Unigramas e Bigramas ---
              precision    recall  f1-score   support

   Não Viral       1.00      1.00      1.00     45897
       Viral       0.17      0.38   

# k. Treinamento, Validação e Teste: Outer K-Fold Cross-Validation;

In [20]:
numeric_features = ['word_count', 'char_count', 'unique_word_ratio', 'avg_word_length']


cols_to_drop = [col for col in numeric_features if col in result_df.columns]
if cols_to_drop:
    print(f"Limpando colunas numéricas pré-existentes para evitar duplicação: {cols_to_drop}")
    result_df = result_df.drop(columns=cols_to_drop)

features_df = result_df['cleaned_text'].apply(lambda x: pd.Series(extract_features(x)))
result_df = pd.concat([result_df, features_df], axis=1)
print("Features numéricas criadas/atualizadas com sucesso.")


print("\nIniciando a Validação Cruzada Aninhada (Outer K-Fold)...")

text_feature = 'cleaned_text'
X = result_df[numeric_features + [text_feature]]
y = result_df['is_viral']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('text', TfidfVectorizer(ngram_range=(1, 2), max_features=5000), text_feature)
    ],
    remainder='drop'
)

full_pipeline = ImbPipeline([
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('classifier', LogisticRegression(solver='saga', random_state=42, max_iter=2000))
])

param_grid = {'classifier__C': [0.1, 1.0, 10]}

outer_cv = KFold(n_splits=5, shuffle=True, random_state=42)
inner_cv = KFold(n_splits=3, shuffle=True, random_state=42)

clf = GridSearchCV(
    estimator=full_pipeline, 
    param_grid=param_grid, 
    cv=inner_cv,
    scoring='f1_macro',
    n_jobs=-1
)

outer_scores = []
for i, (train_idx, test_idx) in enumerate(tqdm(outer_cv.split(X, y), total=outer_cv.get_n_splits(), desc="Outer CV Folds")):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    clf.fit(X_train, y_train)
    
    best_model = clf.best_estimator_
    y_pred = best_model.predict(X_test)
    score = f1_score(y_test, y_pred, pos_label=1)
    outer_scores.append(score)
    
    print(f"\nFold Externo {i+1}: Melhor C={clf.best_params_['classifier__C']}, F1-Score (Viral) = {score:.4f}")

print("\n" + "="*40)
print("Resultado Final da Validação Cruzada Aninhada")
print("="*40)
print(f"F1-Scores para a classe 'Viral' em cada um dos 5 folds: {np.round(outer_scores, 4)}")
print(f"Média do F1-Score: {np.mean(outer_scores):.4f}")
print(f"Desvio Padrão do F1-Score: {np.std(outer_scores):.4f}")

Limpando colunas numéricas pré-existentes para evitar duplicação: ['word_count', 'char_count', 'unique_word_ratio', 'avg_word_length']
Features numéricas criadas/atualizadas com sucesso.

Iniciando a Validação Cruzada Aninhada (Outer K-Fold)...


Outer CV Folds:   0%|          | 0/5 [00:00<?, ?it/s]c:\Users\Pedro\miniforge3\envs\cd\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
Outer CV Folds:  20%|██        | 1/5 [13:42<54:49, 822.38s/it]


Fold Externo 1: Melhor C=10, F1-Score (Viral) = 0.1371


c:\Users\Pedro\miniforge3\envs\cd\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
Outer CV Folds:  40%|████      | 2/5 [27:35<41:25, 828.42s/it]


Fold Externo 2: Melhor C=10, F1-Score (Viral) = 0.1002


...


Fold Externo 3: Melhor C=10, F1-Score (Viral) = 0.0941


c:\Users\Pedro\miniforge3\envs\cd\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
Outer CV Folds:  80%|████████  | 4/5 [54:21<13:31, 811.90s/it]


Fold Externo 4: Melhor C=10, F1-Score (Viral) = 0.1174


c:\Users\Pedro\miniforge3\envs\cd\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
Outer CV Folds: 100%|██████████| 5/5 [1:07:35<00:00, 811.03s/it]


Fold Externo 5: Melhor C=10, F1-Score (Viral) = 0.1438

Resultado Final da Validação Cruzada Aninhada
F1-Scores para a classe 'Viral' em cada um dos 5 folds: [0.1371 0.1002 0.0941 0.1174 0.1438]
Média do F1-Score: 0.1185
Desvio Padrão do F1-Score: 0.0196
